<a href="https://colab.research.google.com/github/2026-2-Generative-AI-Project/Segmentation-SAM/blob/feature%2Fsam-finetune/sam_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM 2.1 파인튜닝

**목표:** 박스 하나를 주면 **그 그릇 하나**(그릇 + 그 위 음식 + 걸친 수저 통째로)를 테두리까지 정확히 따는 SAM.
서비스에서는 후보 그릇을 하나씩 보여 주고, 사용자가 여러 개를 골라 누끼를 만든다. 컵, 따로 놓인 수저는 후보로 나와도 사용자가 안 고르면 되므로 학습하지 않는다.

**라벨링 규칙:** 레포의 `LABELING.md` (정답 = `gt_inst/{이름}.png` 번호 마스크 + `{이름}.json`)

**실행 순서**
1. 라벨링 → 이 노트북 **1-1번**(라벨 검사, `gt/` 흑백 정답 자동 생성)
2. adcanvas_nukki_Fine-tuning **2번**(`split.csv`), **5-1번, 6번**(`boxes.json`, `dino_handoff/candidates.json`) 실행
3. 이 노트북 2번부터 끝까지 → `weights/sam2_ft.pt`
4. adcanvas_nukki_Fine-tuning 3·6·7번 다시 실행 → B3 결과

**학습 방식**
- 이미지 인코더 고정, **마스크 디코더만 학습** (코랩 T4로 충분)
- 박스는 실제 후보 찾기처럼 흔들어서 줌: 정답 박스 / **수저를 놓친 작은 박스** / 실전 박스(DINO 후보, RMBG 박스)
- 가끔 클릭 점(포함 1개, 제외 1개)도 같이 줌 → 서비스의 **클릭 보정** 기능이 계속 잘 되도록
- **박스 밖을 자르지 않음** (박스 밖으로 나간 그릇, 수저 손잡이까지 따야 하므로)
- `split.csv`의 **test 사진은 학습, 모델 선택에 절대 안 씀** (8번에서 마지막에 한 번만 평가)

# 0. 설치

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif

# 1. 경로와 설정
* `CFG`는 adcanvas_nukki_Fine-tuning과 **같은 값** (다르면 경고)
* `FT`가 파인튜닝 설정

In [ ]:
import os, glob, json, time, random, copy, shutil, math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
from scipy import ndimage

register_heif_opener()
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AdCanvas_nukki'
IN_DIR = f'{BASE}/dataset/images'
GT_INST_DIR = f'{BASE}/dataset/gt_inst'          # 그릇별 번호 마스크 + json (LABELING.md)
GT_DIR = f'{BASE}/dataset/gt'                    # 흑백 정답 (지은님 노트북용, 1-1번에서 자동 생성)
RMBG_DIR = f'{BASE}/seg_pipeline/rmbg_handoff'
DINO_JSON = f'{BASE}/seg_pipeline/dino_handoff/candidates.json'   # 1번 노트북 6번이 저장한 DINO 후보
W_DIR = f'{BASE}/weights'
RUN_DIR = f'{W_DIR}/sam_ft_runs'         # 학습 중간 저장, 기록
# 추가 학습 데이터 (선택): 학습에만 쓰고 평가에는 안 씀
EXTRA_IN_DIR = f'{BASE}/extra/images'
EXTRA_GT_INST_DIR = f'{BASE}/extra/gt_inst'
EXTRA_GT_DIR = f'{BASE}/extra/gt'
EXTRA_BOXES = f'{BASE}/extra/rmbg_boxes.json'
os.makedirs(RUN_DIR, exist_ok=True)


def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


device = get_device()

# 지은님 노트북과 같은 값
CFG = dict(
    max_side=2048, rmbg_th=0.5, min_area=0.003, box_pad=0.05, max_boxes=15,
    sam_model='facebook/sam2.1-hiera-small',
    sam_ft_weights=f'{W_DIR}/sam2_ft.pt',
    bf_tol=0.0075, seed=42,
)

# 파인튜닝 설정
FT = dict(
    epochs=30,             # 전체 반복 횟수
    repeats=4,             # 한 epoch에 사진 한 장을 몇 번 쓸지 (매번 박스를 다르게 뽑음)
    lr=5e-5,
    weight_decay=1e-4,
    warmup=0.05,           # 처음 5%는 학습률을 천천히 올림
    train_side=1024,       # 손실 계산 해상도 (긴 변)
    flip=True,             # 좌우 뒤집기 증강
    include_utensils=True, # 걸친 수저를 그릇에 포함 (LABELING.md 규칙)
    real_box_prob=0.4,     # 실전 박스(DINO 후보, RMBG 박스)로 연습할 확률 (그릇 하나와 맞는 박스가 있을 때)
    dish_only_box_prob=0.5,# 수저가 있는 그릇에서 '수저를 놓친 박스'로 연습할 확률
    real_jitter=0.05,      # 실전 박스 흔들기 (박스 크기 대비)
    gt_jitter=0.1,         # 정답 박스 흔들기
    point_prob=0.3,        # 클릭 점을 같이 줄 확률 (클릭 보정 기능 유지)
    min_obj_px=50,         # 학습 해상도에서 이보다 작은 그릇은 무시
    iou_loss_w=0.05,       # SAM 점수 예측 헤드 손실 비중
    grad_clip=1.0,
    val_ratio=0.2,         # 학습 사진이 10장 이상일 때만 검증용으로 떼어둠
    binary_fallback=True,  # gt_inst가 없고 예전 흑백 gt만 있는 사진은 덩어리를 그릇으로 보고 사용
    resume=True,           # 코랩이 끊겨도 이어서 학습
)
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])

EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')


def list_images(d):
    return sorted(p for p in glob.glob(f'{d}/*') if p.lower().endswith(EXTS))


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def load_image(p, max_side=CFG['max_side']):
    img = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    if max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


def boxes_from_mask(m_bin, cfg=CFG):
    """지은님 노트북과 같은 함수: 이진 마스크 → 덩어리마다 박스 [x0, y0, x1, y1] (큰 덩어리부터)"""
    H, W = m_bin.shape
    lab, n = ndimage.label(m_bin)
    if n == 0:
        return []
    sizes = ndimage.sum(m_bin, lab, range(1, n + 1))
    cand = []
    for i, sl in enumerate(ndimage.find_objects(lab)):
        if sizes[i] < cfg['min_area'] * H * W:
            continue
        y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
        ph, pw = (y1 - y0) * cfg['box_pad'], (x1 - x0) * cfg['box_pad']
        cand.append((sizes[i], [max(0, x0 - pw), max(0, y0 - ph), min(W, x1 + pw), min(H, y1 + ph)]))
    cand.sort(key=lambda c: -c[0])
    return [[float(v) for v in b] for _, b in cand[:cfg['max_boxes']]]


def boundary_f1(p, g, tol):
    """지은님 노트북과 같은 경계 점수"""
    pb = p ^ ndimage.binary_erosion(p)
    gb = g ^ ndimage.binary_erosion(g)
    if not pb.any() and not gb.any():
        return 1.0
    if not pb.any() or not gb.any():
        return 0.0
    prec = (ndimage.distance_transform_edt(~gb)[pb] <= tol).mean()
    rec = (ndimage.distance_transform_edt(~pb)[gb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-8))


def mask_box(m):
    ys, xs = np.where(m)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


# boxes.json이 지은님 노트북의 어떤 설정으로 만들어졌는지 확인
BOXES_JSON = f'{RMBG_DIR}/boxes.json'
handoff = json.load(open(BOXES_JSON)) if os.path.exists(BOXES_JSON) else {'cfg': {}, 'images': {}}
for k in ('max_side', 'rmbg_th', 'min_area', 'box_pad', 'max_boxes', 'sam_model'):
    if k in handoff['cfg'] and handoff['cfg'][k] != CFG[k]:
        print(f'경고: {k}가 지은님 노트북({handoff["cfg"][k]})과 다름 ({CFG[k]})')
DINO = json.load(open(DINO_JSON))['images'] if os.path.exists(DINO_JSON) else {}
print('device:', device, '| boxes.json 사진', len(handoff['images']), '장 | DINO 후보 사진', len(DINO), '장')
if not DINO:
    print('DINO 후보가 없음 → 1번 노트북 5-1번, 6번을 먼저 실행하세요 (없으면 RMBG 박스와 정답 박스로만 학습)')

# 1-1. 라벨 검사와 흑백 정답 만들기
* `gt_inst`의 PNG와 json이 `LABELING.md` 형식에 맞는지 검사 (번호 일치, 수저의 `parent`, 사진 회전)
* 문제없는 사진은 `gt/{이름}.png` 흑백 정답(그릇과 수저 전부 흰색)을 자동으로 만듦 → 지은님 노트북 평가와 `split.csv`에 그대로 쓰임
* **라벨을 추가하거나 고칠 때마다 실행**

In [ ]:
def check_labels(img_dir, inst_dir, gt_dir):
    imgs = {stem(p): p for p in list_images(img_dir)}
    ok, problems = [], []
    for pp in sorted(glob.glob(f'{inst_dir}/*.png')):
        n, jp, errs = stem(pp), f'{inst_dir}/{stem(pp)}.json', []
        if n not in imgs:
            errs.append('같은 이름의 원본 사진이 없음')
        if not os.path.exists(jp):
            errs.append('json 없음')
        m = np.asarray(Image.open(pp))
        if m.ndim != 2:
            errs.append(f'흑백 번호 PNG가 아님 (모양 {m.shape})')
        if not errs:
            objs = json.load(open(jp)).get('objects', [])
            ids_png = {int(v) for v in np.unique(m)} - {0}
            ids_json = {int(o['id']) for o in objs}
            if ids_png != ids_json:
                errs.append(f'번호 불일치: png {sorted(ids_png)} / json {sorted(ids_json)}')
            dishes = {int(o['id']) for o in objs if o.get('type') == 'dish'}
            for o in objs:
                if o.get('type') not in ('dish', 'utensil'):
                    errs.append(f"id {o['id']}: type은 dish 또는 utensil")
                elif o['type'] == 'utensil' and o.get('parent') not in dishes:
                    errs.append(f"id {o['id']}: 수저의 parent가 그릇 번호가 아님")
            W, H = ImageOps.exif_transpose(Image.open(imgs[n])).size
            h, w = m.shape
            if abs(w / h - W / H) > 0.02:
                errs.append(f'가로세로 비율이 사진과 다름 (회전 확인): 마스크 {w}x{h} / 사진 {W}x{H}')
        if errs:
            problems.append((n, errs))
        else:
            Image.fromarray(((m > 0) * 255).astype(np.uint8)).save(f'{gt_dir}/{n}.png')
            ok.append(n)
    return ok, problems


for img_dir, inst_dir, gt_dir in [(IN_DIR, GT_INST_DIR, GT_DIR), (EXTRA_IN_DIR, EXTRA_GT_INST_DIR, EXTRA_GT_DIR)]:
    if not os.path.isdir(inst_dir):
        continue
    os.makedirs(gt_dir, exist_ok=True)
    ok, problems = check_labels(img_dir, inst_dir, gt_dir)
    print(f'{inst_dir}: 통과 {len(ok)}장 → {gt_dir}에 흑백 정답 저장 | 문제 {len(problems)}장')
    for n, errs in problems:
        print('  -', n, ':', ' / '.join(errs))

# 2. 학습 데이터 정하기
* 시험/학습 구분은 **`split.csv`가 기준**. test는 학습에서 빠지고 8번 평가에만 쓰임
* 정답은 `gt_inst`(그릇별)를 우선 쓰고, 없으면 예전 흑백 `gt`를 덩어리별로 나눠서 씀 (붙은 그릇이 하나로 묶이므로 임시용)
* `extra/` 폴더 데이터는 학습에만 추가

In [ ]:
split = pd.read_csv(f'{BASE}/dataset/split.csv')
test_names = set(split[split.split == 'test'].image)
train_names = set(split[split.split == 'train'].image)


def label_of(n, inst_dir, gt_dir):
    if os.path.exists(f'{inst_dir}/{n}.png') and os.path.exists(f'{inst_dir}/{n}.json'):
        return {'kind': 'inst', 'mask': f'{inst_dir}/{n}.png', 'json': f'{inst_dir}/{n}.json'}
    if FT['binary_fallback'] and os.path.exists(f'{gt_dir}/{n}.png'):
        return {'kind': 'binary', 'mask': f'{gt_dir}/{n}.png', 'json': None}
    return None


own_paths = {stem(p): p for p in list_images(IN_DIR)}
samples, test_samples, not_in_split = [], [], []
for n, p in own_paths.items():
    lab = label_of(n, GT_INST_DIR, GT_DIR)
    if lab is None:
        continue
    rec = {'name': n, 'img': p, 'src': 'own', **lab}
    if n in test_names:
        test_samples.append(rec)
    elif n in train_names:
        samples.append(rec)
    else:
        not_in_split.append(n)
if not_in_split:
    print('split.csv에 아직 없는 정답 사진 (이번 학습에서 제외 → 지은님 노트북 2번 셀 실행 후 다시):', not_in_split)

extra_handoff = json.load(open(EXTRA_BOXES)) if os.path.exists(EXTRA_BOXES) else {'images': {}}
for p in list_images(EXTRA_IN_DIR):
    n = stem(p)
    if n in test_names or n in own_paths:
        continue
    lab = label_of(n, EXTRA_GT_INST_DIR, EXTRA_GT_DIR)
    if lab:
        samples.append({'name': n, 'img': p, 'src': 'extra', **lab})

assert not ({s['name'] for s in samples} & test_names), 'test 사진이 학습에 들어감!'

rng = random.Random(CFG['seed'])
order = samples[:]
rng.shuffle(order)
n_val = int(len(order) * FT['val_ratio']) if len(order) >= 10 else 0
val_samples, train_samples = order[:n_val], order[n_val:]

kinds = pd.Series([s['kind'] for s in train_samples]).value_counts().to_dict()
print(f'학습 {len(train_samples)}장 {kinds} | 검증 {len(val_samples)}장 | test {len(test_samples)}장 (8번에서만 사용)')
if kinds.get('binary'):
    print('주의: 흑백 gt로 학습하는 사진은 붙은 그릇이 하나로 묶임. gt_inst로 다시 라벨링하면 더 정확해짐')
if len(train_samples) < 20:
    print('주의: 학습 사진이 적어서 외우기 쉬움. 라벨링을 더 하거나 extra/ 폴더에 데이터를 넣는 걸 추천')

## 2-1. (선택) 추가 데이터의 RMBG 박스 만들기
* `extra/` 사진에도 실제 RMBG 박스로 연습시키고 싶을 때만 실행 (`HF_TOKEN` 필요)
* 한 번 만들면 `extra/rmbg_boxes.json`에 저장되어 다음부터는 건너뜀

In [ ]:
todo = [s for s in samples if s['src'] == 'extra' and s['name'] not in extra_handoff['images']]
print('RMBG 박스가 없는 추가 사진:', len(todo), '장')
if todo:
    !pip install -q timm kornia einops transformers
    from google.colab import userdata
    from huggingface_hub import login
    from torchvision import transforms
    from transformers import AutoModelForImageSegmentation

    login(token=userdata.get('HF_TOKEN'))
    rmbg = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True).to(device).eval()
    rmbg_tf = transforms.Compose([transforms.Resize((1024, 1024)), transforms.ToTensor(),
                                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    for i, s in enumerate(todo):
        img = load_image(s['img'])
        with torch.no_grad():
            x = rmbg_tf(img).unsqueeze(0).to(device, next(rmbg.parameters()).dtype)
            pred = rmbg(x)[-1].sigmoid()[0, 0].float().cpu()
        soft = np.asarray(transforms.functional.to_pil_image(pred).resize(img.size, Image.BILINEAR), np.float32) / 255
        boxes = boxes_from_mask(soft > CFG['rmbg_th'])
        W, H = img.size
        extra_handoff['images'][s['name']] = {'size': [W, H], 'boxes': boxes,
                                              'boxes_norm': [[b[0] / W, b[1] / H, b[2] / W, b[3] / H] for b in boxes]}
        if (i + 1) % 20 == 0:
            print(i + 1, '/', len(todo))
    json.dump({'cfg': CFG, 'images': extra_handoff['images']}, open(EXTRA_BOXES, 'w'), ensure_ascii=False, indent=1)
    del rmbg
    if device == 'cuda':
        torch.cuda.empty_cache()
    print('저장:', EXTRA_BOXES)

# 3. SAM 불러오기와 사진 준비
* 인코더는 고정이라 사진마다 특징을 **한 번만** 계산해서 저장 → 학습이 매우 빨라짐
* 사진마다 그릇 목록을 만듦: 그릇 정답(그릇 + 음식 + 걸친 수저), 정답 박스, 수저를 뺀 박스, 그 그릇과 맞는 실전 박스(DINO 후보, RMBG 박스)
* 실전 박스가 그릇 여러 개를 한꺼번에 감싸면(붙은 그릇) 그 박스는 학습에 안 씀 → 이 문제는 후보 찾기 단계에서 해결
* 디코더는 `eval()` 상태로 학습: 추론과 똑같은 방식으로 마스크를 고르게 하기 위함 (디코더에는 dropout이 없음)

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

predictor = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)
model = predictor.model
model.eval()
model.requires_grad_(False)
dec = model.sam_mask_decoder
dec.requires_grad_(True)
init_dec = copy.deepcopy(dec.state_dict())      # 파인튜닝 전 디코더 (비교용)
print('학습하는 파라미터:', round(sum(p.numel() for p in dec.parameters()) / 1e6, 2), 'M')


def read_label(rec, size):
    """정답 → (번호 배열, 물체 목록). size = (w, h)"""
    m = Image.open(rec['mask'])
    if rec['kind'] == 'inst':
        arr = np.asarray(m.resize(size, Image.NEAREST)).astype(np.int32)
        objs = json.load(open(rec['json']))['objects']
    else:   # 예전 흑백 gt: 떨어진 덩어리를 각각 그릇으로
        arr, n = ndimage.label(np.asarray(m.convert('L').resize(size, Image.NEAREST)) > 127)
        objs = [{'id': i, 'type': 'dish'} for i in range(1, n + 1)]
    return arr, objs


def real_boxes_for(rec, W, H):
    """실전 박스: DINO 그릇 후보(1번 노트북 6번) + RMBG 덩어리 박스 → [(박스, 'dino' 또는 'rmbg')] (지금 크기 W, H 기준)"""
    out = []
    dn = DINO.get(rec['name']) if rec['src'] == 'own' else None
    if dn:
        sx, sy = W / dn['size'][0], H / dn['size'][1]
        out += [([c['box'][0] * sx, c['box'][1] * sy, c['box'][2] * sx, c['box'][3] * sy], 'dino')
                for c in dn['candidates'] if c['dish']]
    src = handoff if rec['src'] == 'own' else extra_handoff
    r = src['images'].get(rec['name'])
    if r:
        out += [([b[0] * W, b[1] * H, b[2] * W, b[3] * H], 'rmbg') for b in r['boxes_norm']]
    return out


@torch.no_grad()
def embed(img_np):
    predictor.set_image(img_np)
    f = predictor._features
    return f['image_embed'].half().cpu(), [h.half().cpu() for h in f['high_res_feats']]


def make_variant(img, rec, rboxes, flip):
    W, H = img.size
    s = FT['train_side'] / max(W, H)
    w, h = round(W * s), round(H * s)
    arr, objs = read_label(rec, (w, h))
    if flip:
        img, arr = ImageOps.mirror(img), np.ascontiguousarray(arr[:, ::-1])
    dishes = []
    for o in objs:
        if o['type'] != 'dish':
            continue
        ids = [o['id']] + ([u['id'] for u in objs if u['type'] == 'utensil' and u.get('parent') == o['id']]
                           if FT['include_utensils'] else [])
        full, dish_only = np.isin(arr, ids), arr == o['id']
        if full.sum() < FT['min_obj_px']:
            continue
        dishes.append({'ids': ids, 'area': int(full.sum()), 'box': mask_box(full),
                       'dish_box': mask_box(dish_only) if dish_only.any() else mask_box(full),
                       'has_utensil': len(ids) > 1, 'real': []})
    # 실전 박스(DINO, RMBG) 중 그릇 하나와만 맞는 것 (그 그릇의 50% 이상을 감싸고, 다른 그릇은 50% 미만)
    if rboxes:
        for b, src in rboxes:
            if flip:
                b = [W - b[2], b[1], W - b[0], b[3]]
            b = [v * s for v in b]
            crop = arr[int(b[1]):int(math.ceil(b[3])), int(b[0]):int(math.ceil(b[2]))]
            cnt = np.bincount(crop.ravel(), minlength=arr.max() + 1)
            cover = [sum(cnt[i] for i in d['ids'] if i < len(cnt)) / d['area'] for d in dishes]
            hit = [i for i, c in enumerate(cover) if c >= 0.5]
            if len(hit) == 1:
                dishes[hit[0]]['real'].append((b, src))
    emb, hr = embed(np.array(img))
    return {'hw': (h, w), 'inst': arr.astype(np.int16), 'dishes': dishes, 'emb': emb, 'hr': hr}


def build_items(sample_list, flip=FT['flip']):
    items = []
    for i, rec in enumerate(sample_list):
        img = load_image(rec['img'])
        rboxes = real_boxes_for(rec, *img.size)
        vs = [make_variant(img, rec, rboxes, False)] + ([make_variant(img, rec, rboxes, True)] if flip else [])
        items.append({'name': rec['name'], 'variants': vs})
        if (i + 1) % 20 == 0:
            print(i + 1, '/', len(sample_list))
    return items


t0 = time.time()
train_items = build_items(train_samples)
val_items = build_items(val_samples, flip=False)
d_all = [d for it in train_items for d in it['variants'][0]['dishes']]
print(f'준비 완료 {time.time() - t0:.0f}초 | 학습 그릇 {len(d_all)}개 '
      f'(수저 걸친 그릇 {sum(d["has_utensil"] for d in d_all)}, 실전 박스와 맞는 그릇: DINO {sum(any(s == "dino" for _, s in d["real"]) for d in d_all)}, RMBG {sum(any(s == "rmbg" for _, s in d["real"]) for d in d_all)})')

# 4. 연습 문제 만들기, 손실, 평가
* **박스 고르기** (그릇마다): 실전 박스(DINO 후보, RMBG 박스가 있으면) / 수저를 놓친 작은 박스(수저 있으면) / 정답 박스를 흔든 것
* **정답**: 박스 크기와 상관없이 항상 그 그릇 전체 (그릇 + 음식 + 걸친 수저)
* **평가**: 그릇마다 IoU와 경계 점수(BF)를 박스 종류별로
  * `정답 박스`: 후보 찾기가 잘 됐을 때
  * `수저 놓친 박스`: 박스가 수저 손잡이를 놓쳤을 때 (박스 밖까지 따야 함)
  * `DINO 박스`: 지금 파이프라인(C3)의 실제 박스
  * `RMBG 박스`: 예전 파이프라인(B2)의 박스

In [ ]:
def jitter(b, amt, h, w):
    x0, y0, x1, y1 = b
    bw, bh = x1 - x0, y1 - y0
    x0 += np.random.uniform(-amt, amt) * bw; x1 += np.random.uniform(-amt, amt) * bw
    y0 += np.random.uniform(-amt, amt) * bh; y1 += np.random.uniform(-amt, amt) * bh
    x0, x1 = np.clip(sorted([x0, x1]), 0, w)
    y0, y1 = np.clip(sorted([y0, y1]), 0, h)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return [float(v) for v in b]
    return [float(x0), float(y0), float(x1), float(y1)]


def pick_box(d, h, w):
    if d['real'] and np.random.rand() < FT['real_box_prob']:
        return jitter(random.choice(d['real'])[0], FT['real_jitter'], h, w)
    if d['has_utensil'] and np.random.rand() < FT['dish_only_box_prob']:
        return jitter(d['dish_box'], FT['gt_jitter'], h, w)
    return jitter(d['box'], FT['gt_jitter'], h, w)


def pick_points(target, box):
    """클릭 보정 연습: 그릇 안 1점(포함), 박스 안이지만 그릇 밖 1점(제외). 없으면 빈 점(-1)"""
    pts, lab = [[0.0, 0.0], [0.0, 0.0]], [-1, -1]
    ys, xs = np.nonzero(target)
    if len(xs):
        k = np.random.randint(len(xs)); pts[0], lab[0] = [float(xs[k]), float(ys[k])], 1
    x0, y0, x1, y1 = [int(v) for v in box]
    out = ~target[y0:y1, x0:x1]
    ys, xs = np.nonzero(out)
    if len(xs):
        k = np.random.randint(len(xs)); pts[1], lab[1] = [float(xs[k] + x0), float(ys[k] + y0)], 0
    return pts, lab


def decode(v, boxes, points=None, plabels=None):
    """캐시한 특징 + 박스(+점) → 마스크 로짓 (B, h, w), 점수 예측 (B,)
    SAM2ImagePredictor._predict와 같은 계산 (no_grad만 뺀 것)"""
    emb = v['emb'].to(device).float()
    hr = [x.to(device).float() for x in v['hr']]
    b = torch.tensor(boxes, dtype=torch.float32, device=device)
    coords = predictor._transforms.transform_boxes(b, normalize=True, orig_hw=v['hw'])
    labels = torch.tensor([[2, 3]], dtype=torch.int, device=device).repeat(len(boxes), 1)
    if points is not None:
        pc = predictor._transforms.transform_coords(
            torch.tensor(points, dtype=torch.float32, device=device), normalize=True, orig_hw=v['hw'])
        coords = torch.cat([coords, pc], 1)
        labels = torch.cat([labels, torch.tensor(plabels, dtype=torch.int, device=device)], 1)
    sparse, dense = model.sam_prompt_encoder(points=(coords, labels), boxes=None, masks=None)
    low, iou_pred, _, _ = dec(
        image_embeddings=emb, image_pe=model.sam_prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse, dense_prompt_embeddings=dense,
        multimask_output=False, repeat_image=len(boxes) > 1, high_res_features=hr)
    logits = F.interpolate(low, size=v['hw'], mode='bilinear', align_corners=False)[:, 0]
    return logits, iou_pred[:, 0]


def seg_loss(logits, target, iou_pred):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = logits.sigmoid().flatten(1)
    t = target.flatten(1)
    dice = 1 - (2 * (p * t).sum(1) + 1) / (p.sum(1) + t.sum(1) + 1)
    with torch.no_grad():
        pb = (logits > 0).float().flatten(1)
        true_iou = (pb * t).sum(1) / ((pb + t) > 0).float().sum(1).clamp(min=1)
    return bce + dice.mean() + FT['iou_loss_w'] * F.mse_loss(iou_pred.float(), true_iou)


EVAL_KINDS = {'정답 박스': 'box', '수저 놓친 박스': 'dish_box', 'DINO 박스': 'dino', 'RMBG 박스': 'rmbg'}


@torch.no_grad()
def evaluate(items):
    """그릇마다 IoU, BF를 박스 종류별로. 반환: (표, 대표 점수)"""
    rows = []
    for it in items:
        v = it['variants'][0]
        tol = CFG['bf_tol'] * math.hypot(*v['hw'])
        for kind, key in EVAL_KINDS.items():
            pairs = []
            for d in v['dishes']:
                if key == 'box':
                    pairs.append((d, d['box']))
                elif key == 'dish_box' and d['has_utensil']:
                    pairs.append((d, d['dish_box']))
                elif key in ('dino', 'rmbg'):
                    rb = [b for b, s in d['real'] if s == key]
                    if rb:
                        pairs.append((d, rb[0]))
            if not pairs:
                continue
            logits, _ = decode(v, [b for _, b in pairs])
            preds = (logits > 0).cpu().numpy()
            for (d, _), p in zip(pairs, preds):
                g = np.isin(v['inst'], d['ids'])
                u = (p | g).sum()
                rows.append({'image': it['name'], 'kind': kind,
                             'IoU': (p & g).sum() / u if u else 1.0, 'BF': boundary_f1(p, g, tol)})
    df = pd.DataFrame(rows)
    if df.empty:
        return df, float('nan')
    table = df.groupby('kind')[['IoU', 'BF']].mean().reindex([k for k in EVAL_KINDS if k in set(df.kind)])
    table['그릇 수'] = df.groupby('kind').size()
    score = float(table[['IoU', 'BF']].mean(axis=1).mean())   # 박스 종류별 (IoU+BF)/2 의 평균
    return table, score


eval_items = val_items if val_items else train_items
eval_name = '검증' if val_items else '학습(참고용)'
base_table, base_score = evaluate(eval_items)
print(f'파인튜닝 전 {eval_name} 점수 {base_score:.4f}')
print(base_table.round(4))

# 5. 학습
* 매 epoch마다 `weights/sam_ft_runs/last.pt`에 저장 → 코랩이 끊기면 이 셀만 다시 실행하면 이어서 학습
* 검증 사진이 있으면 검증 점수가 가장 좋은 디코더를 최종으로 씀

In [ ]:
params = [p for p in dec.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=FT['lr'], weight_decay=FT['weight_decay'])
steps_per_epoch = max(1, len(train_items) * FT['repeats'])
total_steps = FT['epochs'] * steps_per_epoch
warm = max(1, int(total_steps * FT['warmup']))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))

LAST = f'{RUN_DIR}/last.pt'
start_ep, history = 0, []
best = {'score': base_score, 'state': copy.deepcopy(dec.state_dict()), 'epoch': 0}
if FT['resume'] and os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device, weights_only=False)   # 우리가 저장한 파일
    if ck.get('n_train') == len(train_items) and ck.get('total_steps') == total_steps:
        dec.load_state_dict(ck['dec']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        start_ep, history, best = ck['epoch'], ck['history'], ck['best']
        print(f'{start_ep} epoch부터 이어서 학습')
    else:
        print('데이터나 설정이 바뀌어서 처음부터 학습')

for ep in range(start_ep, FT['epochs']):
    t0 = time.time()
    order = [i for i in range(len(train_items)) for _ in range(FT['repeats'])]
    random.shuffle(order)
    losses = []
    for i in order:
        v = random.choice(train_items[i]['variants'])
        ds = random.sample(v['dishes'], min(len(v['dishes']), CFG['max_boxes']))
        if not ds:
            continue
        h, w = v['hw']
        boxes = [pick_box(d, h, w) for d in ds]
        targets = np.stack([np.isin(v['inst'], d['ids']) for d in ds])
        points = plabels = None
        if np.random.rand() < FT['point_prob']:
            pl = [pick_points(t, b) for t, b in zip(targets, boxes)]
            points, plabels = [p for p, _ in pl], [l for _, l in pl]
        logits, iou_pred = decode(v, boxes, points, plabels)
        loss = seg_loss(logits, torch.from_numpy(targets).float().to(device), iou_pred)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, FT['grad_clip'])
        opt.step(); sched.step()
        losses.append(loss.item())

    table, score = evaluate(eval_items)
    history.append({'epoch': ep + 1, 'loss': float(np.mean(losses)) if losses else float('nan'),
                    'score': score, **{f'IoU {k}': float(v) for k, v in table['IoU'].items()},
                    'sec': round(time.time() - t0, 1)})
    if val_items and score > best['score']:
        best = {'score': score, 'state': copy.deepcopy(dec.state_dict()), 'epoch': ep + 1}
    ious = ' | '.join(f'{k} {v:.3f}' for k, v in table['IoU'].items())
    print(f"[{ep + 1}/{FT['epochs']}] loss {history[-1]['loss']:.4f} | {eval_name} 점수 {score:.4f} ({ious}) | {history[-1]['sec']}초")
    torch.save({'dec': dec.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': ep + 1, 'history': history, 'best': best,
                'n_train': len(train_items), 'total_steps': total_steps}, LAST)

hist = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.epoch, hist.loss); ax[0].set_title('train loss')
for c in [c for c in hist.columns if c.startswith('IoU')]:
    ax[1].plot(hist.epoch, hist[c], label=c)
ax[1].set_title(f'{eval_name} IoU'); ax[1].legend()
plt.tight_layout(); plt.show()

# 6. 저장 ('adcanvas_nukki_Fine-tuning'이 바로 불러가는 형식)
* `weights/sam2_ft.pt`: **SAM 전체 state_dict** → adcanvas_nukki_Fine-tuning에서 `없는 키 0 | 남는 키 0`이 나와야 정상
* 같은 파일을 `sam_ft_runs/`에 날짜별로도 남김 (설정, 학습 기록 json 포함)

In [ ]:
if val_items:
    dec.load_state_dict(best['state'])
    print(f"검증 점수가 가장 좋은 {best['epoch']} epoch 디코더 사용 (점수 {best['score']:.4f})")

final_table, final_score = evaluate(eval_items)
stamp = time.strftime('%Y%m%d-%H%M')
run_path = f'{RUN_DIR}/sam2_ft_{stamp}.pt'
torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, run_path)
shutil.copy(run_path, CFG['sam_ft_weights'])
info = {'saved': stamp, 'sam_model': CFG['sam_model'], 'cfg': CFG, 'ft': FT,
        'train': [s['name'] for s in train_samples], 'val': [s['name'] for s in val_samples],
        'eval_on': eval_name, 'base_score': base_score, 'final_score': final_score,
        'best_epoch': best['epoch'] if val_items else FT['epochs'], 'history': history}
json.dump(info, open(f'{RUN_DIR}/sam2_ft_{stamp}.json', 'w'), ensure_ascii=False, indent=1)
print('저장:', CFG['sam_ft_weights'], f'({os.path.getsize(run_path) / 1e6:.0f}MB)')
print(pd.concat({'전': base_table[['IoU', 'BF']], '후': final_table[['IoU', 'BF']]}, axis=1).round(4))

# 7. 확인
* adcanvas_nukki_Fine-tuning과 **똑같은 방법**으로 불러와서 키가 다 맞는지 확인
* 검증(없으면 학습) 사진으로 그릇별 전후 비교 (그릇마다 다른 색)

In [ ]:
check = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)
res = check.model.load_state_dict(torch.load(CFG['sam_ft_weights'], map_location=device), strict=False)
print('없는 키', len(res.missing_keys), '| 남는 키', len(res.unexpected_keys))
assert not res.missing_keys and not res.unexpected_keys, '키가 안 맞음 → sam_model 이름이 지은님 노트북과 같은지 확인'
same = all(torch.equal(a.cpu(), b.cpu()) for a, b in zip(check.model.sam_mask_decoder.state_dict().values(),
                                                         dec.state_dict().values()))
print('디코더 가중치 일치:', same)
del check

COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)])


def paint(img_np, masks, boxes=None):
    out = img_np.copy()
    for k, m in enumerate(masks):
        c = COLORS[k % len(COLORS)]
        out[m] = (out[m] * 0.45 + c * 0.55).astype(np.uint8)
    return out


@torch.no_grad()
def predict_dishes(v, state, key='box'):
    dec.load_state_dict(state)
    boxes = [d[key] for d in v['dishes']]
    return list((decode(v, boxes)[0] > 0).cpu().numpy()) if boxes else [], boxes


final_state = copy.deepcopy(dec.state_dict())
show = (val_items or train_items)[:4]
smp_map = {s['name']: s for s in samples}
fig, axes = plt.subplots(len(show), 3, figsize=(13, 4 * len(show)), squeeze=False)
for r, it in enumerate(show):
    v = it['variants'][0]
    img = np.array(load_image(smp_map[it['name']]['img']).resize(v['hw'][::-1]))
    gts = [np.isin(v['inst'], d['ids']) for d in v['dishes']]
    for c, (title, ms) in enumerate([('GT', gts), ('before', predict_dishes(v, init_dec)[0]),
                                     ('after', predict_dishes(v, final_state)[0])]):
        axes[r][c].imshow(paint(img, ms))
        for x0, y0, x1, y1 in [d['box'] for d in v['dishes']]:
            axes[r][c].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='w', lw=1))
        axes[r][c].set_title(f"{it['name']} {title}"); axes[r][c].axis('off')
dec.load_state_dict(final_state)
plt.tight_layout(); plt.show()

# 8. test 평가 (보고용)
* **학습과 모델 선택이 다 끝난 뒤에 한 번만** 실행. 이 결과를 보고 설정을 고쳐서 다시 학습하면 test가 의미를 잃음
* 그릇별 IoU, BF를 박스 종류별로 파인튜닝 전후 비교 → `sam_ft_runs/test_{날짜}.csv`
* (지은님 노트북 7번은 사진 전체를 한 덩어리로 보는 점수, 여기는 그릇 하나하나의 점수)

In [ ]:
if test_samples:
    test_items = build_items(test_samples, flip=False)
    dec.load_state_dict(init_dec)
    t_before, _ = evaluate(test_items)
    dec.load_state_dict(final_state)
    t_after, _ = evaluate(test_items)
    tt = pd.concat({'전': t_before[['IoU', 'BF']], '후': t_after[['IoU', 'BF']]}, axis=1)
    tt['그릇 수'] = t_after['그릇 수']
    tt.to_csv(f'{RUN_DIR}/test_{stamp}.csv', encoding='utf-8-sig')
    print(tt.round(4))
else:
    print('test 사진이 없음')

# 9. 파이프라인에 넣는 함수
* 서비스 흐름: **후보 박스 → 그릇마다 마스크(합치지 않음) → 사용자가 고름 → 클릭 보정 → 고른 그릇만 누끼**
* `set_image`는 사진당 한 번만 → 클릭 보정은 디코더만 다시 돌아서 거의 즉시 반응
* **박스 밖을 자르지 않음**

In [ ]:
# 파인튜닝 SAM은 단일 마스크 출력(multimask_output=False)으로 학습했으므로, 추론도 단일 마스크로 해야 학습 결과가 쓰임
@torch.no_grad()
def sam_candidates(predictor, img_np, boxes):
    """후보 박스들 → 그릇 후보 목록 [{'id', 'box', 'mask'(bool HxW), 'score'}]"""
    if not boxes:
        return []
    predictor.set_image(img_np)
    H, W = img_np.shape[:2]
    masks, scores, _ = predictor.predict(box=np.array(boxes, dtype=np.float32), multimask_output=False)
    masks = np.asarray(masks).reshape(-1, H, W) > 0.5
    scores = np.asarray(scores).reshape(-1)
    return [{'id': i, 'box': b, 'mask': m, 'score': float(s)} for i, (b, m, s) in enumerate(zip(boxes, masks, scores))]


@torch.no_grad()
def sam_refine_click(predictor, box, points, labels):
    """클릭 보정: 같은 사진에서 박스 + 클릭 점(1=포함, 0=제외) → 새 마스크. set_image를 다시 하지 않음"""
    m, s, _ = predictor.predict(point_coords=np.array(points, np.float32), point_labels=np.array(labels, np.int32),
                                box=np.array(box, np.float32), multimask_output=False)
    return np.asarray(m)[0] > 0.5


def merge_selected(cands, selected_ids):
    """사용자가 고른 그릇들 → 마스크 하나 (한상차림처럼 여러 그릇 선택)"""
    H, W = cands[0]['mask'].shape
    out = np.zeros((H, W), bool)
    for c in cands:
        if c['id'] in selected_ids:
            out |= c['mask']
    return out


# 예시: 검증(없으면 학습) 사진 한 장
rec = (val_samples or train_samples)[0]
img = np.array(load_image(rec['img']))
real = real_boxes_for(rec, img.shape[1], img.shape[0])
boxes = [b for b, s in real if s == 'dino'] or [b for b, s in real]
ft_pred = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)
ft_pred.model.load_state_dict(torch.load(CFG['sam_ft_weights'], map_location=device), strict=False)
cands = sam_candidates(ft_pred, img, boxes)
print(f"{rec['name']}: 후보 {len(cands)}개", [round(c['score'], 3) for c in cands])
if cands:
    plt.figure(figsize=(6, 6)); plt.imshow(paint(img, [c['mask'] for c in cands])); plt.axis('off'); plt.show()
del ft_pred